# Formula-match retrieval — publication figures

Top-k accuracy vs. k, one figure per (comparison × ranking metric), mean ±
95% CI (t-distribution) shaded band across 3 seeds. Three comparisons:
- **scaffold**: full scaffold-split test set, ICICLE vs. NEIMS/RASSP/MassFormer
- **rassp_subset**: RASSP's native scaffold split (head-to-head subset), same
  4 models — ICICLE/NEIMS/MassFormer downsampled from their full-scaffold
  results to RASSP's covered molecules (RASSP has no candidate pool for the
  rest), RASSP used unfiltered.
- **random**: random split, ICICLE-only (baselines not yet trained)

## Set style utils

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from icicle.utils.visualization.eval_plots import (
    load_seed_csvs,
    plot_topk_curves,
)
from icicle.utils.visualization.style import save_fig, set_style

set_style("manuscript")

RESULTS = Path("/home/magled/icicle-dev/results/eval")
OUTPUT_DIR = Path("figures/retrieval_formula")
K_VALUES = [1, 2, 3, 4, 5, 10, 15, 20, 30, 40, 50]
METRICS = [
    "cosine_similarity",
    "entropy_similarity",
    "weighted_cosine_nist_gc",
    "composite_similarity_nist_gc",
]
METRIC_DISPLAY_NAMES = {
    "cosine_similarity": "Cosine Similarity",
    "entropy_similarity": "Entropy Similarity",
    "weighted_cosine_nist_gc": "Weighted Cosine",
    "composite_similarity_nist_gc": "Composite Similarity",
}

## Configuration

`COMPARISON_MODELS` maps comparison name → model label → per-seed
`retrieval_with_formula_results.csv` paths.

In [ ]:
COMPARISON_MODELS = {
    "scaffold": {
        "ICICLE": [
            RESULTS
            / f"final_entropy_scaffold_s{i}_retr"
            / "retrieval_with_formula_results.csv"
            for i in (1, 2, 3)
        ],
        "NEIMS": [
            RESULTS
            / f"neims_scaffold_s{i}"
            / "retrieval_with_formula_results.csv"
            for i in (1, 2, 3)
        ],
        "RASSP": [
            RESULTS
            / f"rassp_scaffold_s{i}"
            / "retrieval_with_formula_results.csv"
            for i in (1, 2, 3)
        ],
        "MassFormer": [
            RESULTS
            / f"massformer_scaffold_s{i}"
            / "retrieval_with_formula_results.csv"
            for i in (1, 2, 3)
        ],
    },
    "scaffold_rassp_subset": {
        # RASSP-native-split subset: ICICLE/NEIMS/MassFormer downsampled from
        # their full-scaffold eval CSVs to RASSP's covered molecules (see
        # examples/scripts/evaluation/filter_to_rassp_subset.py); RASSP itself
        # already ran on this exact split, used unfiltered.
        "ICICLE": [
            RESULTS
            / "rassp_subset_comparison"
            / f"icicle_scaffold_s{i}_retr_rassp_subset.csv"
            for i in (1, 2, 3)
        ],
        "NEIMS": [
            RESULTS
            / "rassp_subset_comparison"
            / f"neims_scaffold_s{i}_retr_rassp_subset.csv"
            for i in (1, 2, 3)
        ],
        "RASSP": [
            RESULTS
            / f"rassp_scaffold_s{i}"
            / "retrieval_with_formula_results.csv"
            for i in (1, 2, 3)
        ],
        "MassFormer": [
            RESULTS
            / "rassp_subset_comparison"
            / f"massformer_scaffold_s{i}_retr_rassp_subset.csv"
            for i in (1, 2, 3)
        ],
    },
    "random": {
        "ICICLE": [
            RESULTS
            / f"final_entropy_random_s{i}_retr"
            / "retrieval_with_formula_results.csv"
            for i in (1, 2, 3)
        ],
        "NEIMS": [
            RESULTS
            / f"neims_random_s{i}"
            / "retrieval_with_formula_results.csv"
            for i in (1, 2, 3)
        ],
        "RASSP": [
            RESULTS
            / f"rassp_random_s{i}"
            / "retrieval_with_formula_results.csv"
            for i in (1, 2, 3)
        ],
        "MassFormer": [
            RESULTS
            / f"massformer_random_s{i}"
            / "retrieval_with_formula_results.csv"
            for i in (1, 2, 3)
        ],
    },
    "random_rassp_subset": {
        # Same idea as "scaffold_rassp_subset" but for the random split:
        # ICICLE/NEIMS/MassFormer downsampled to RASSP's random-native split
        # coverage; RASSP itself already ran on this exact split, unfiltered.
        "ICICLE": [
            RESULTS
            / "rassp_subset_comparison"
            / f"icicle_random_s{i}_retr_rassp_subset.csv"
            for i in (1, 2, 3)
        ],
        "NEIMS": [
            RESULTS
            / "rassp_subset_comparison"
            / f"neims_random_s{i}_retr_rassp_subset.csv"
            for i in (1, 2, 3)
        ],
        "RASSP": [
            RESULTS
            / f"rassp_random_s{i}"
            / "retrieval_with_formula_results.csv"
            for i in (1, 2, 3)
        ],
        "MassFormer": [
            RESULTS
            / "rassp_subset_comparison"
            / f"massformer_random_s{i}_retr_rassp_subset.csv"
            for i in (1, 2, 3)
        ],
    },
}

## Load + plot

One figure per (comparison × metric), plus a `no_rassp` variant (all models
minus RASSP) whenever RASSP is present in that comparison. A model is
dropped if none of its seed CSVs exist yet.


In [ ]:
loaded_by_comparison = {}
for comparison, models in COMPARISON_MODELS.items():
    model_dfs = {
        label: load_seed_csvs(paths) for label, paths in models.items()
    }
    for label, dfs in model_dfs.items():
        print(f"[{comparison}] {label}: {len(dfs)}/3 seeds loaded")
    model_dfs = {label: dfs for label, dfs in model_dfs.items() if dfs}
    loaded_by_comparison[comparison] = model_dfs

## Top-k accuracy curves

One figure per (comparison × ranking metric). All models on the same axes,
shaded 95% CI band.

In [ ]:
for comparison, model_dfs in loaded_by_comparison.items():
    if "ICICLE" not in model_dfs:
        print(
            f"[{comparison}] ICICLE has no results yet, skipping comparison entirely"
        )
        continue
    for metric in METRICS:
        rank_col = f"rank_{metric}"
        dfs_with_metric = {
            k: v for k, v in model_dfs.items() if rank_col in v[0].columns
        }
        if "ICICLE" not in dfs_with_metric:
            continue

        variants = {"": dfs_with_metric}
        if "RASSP" in dfs_with_metric:
            variants["no_rassp"] = {
                k: v for k, v in dfs_with_metric.items() if k != "RASSP"
            }

        for variant_suffix, variant_dfs in variants.items():
            fig = plot_topk_curves(variant_dfs, rank_col, K_VALUES)
            fig.axes[0].set_xlabel("Top-k")
            name_parts = [f"retrieval_formula_topk_{metric}_{comparison}"]
            if variant_suffix:
                name_parts.append(variant_suffix)
            save_fig(fig, "_".join(name_parts), OUTPUT_DIR)
            plt.show()
            plt.close(fig)

## Summary table (mean ± 95% CI, MRR, median rank)

In [ ]:
import numpy as np

from icicle.utils.visualization.eval_plots import compute_topk_curve, mean_ci_t


def _query_col(df):
    for col in ("query_inchikey14", "query_mol_id", "spec"):
        if col in df.columns:
            return col
    raise KeyError(f"No query column found in {list(df.columns)}")


def _correct_ranks(df, rank_col):
    if "is_decoy" in df.columns:
        correct = df[~df["is_decoy"]]
    else:
        correct = df[df["is_correct"]]
    return correct.groupby(_query_col(df))[rank_col].min()


SUMMARY_K_VALUES = [1, 5, 10, 20, 50]

rows = []
for comparison, model_dfs in loaded_by_comparison.items():
    for metric in METRICS:
        rank_col = f"rank_{metric}"
        for label, dfs in model_dfs.items():
            dfs_with_col = [df for df in dfs if rank_col in df.columns]
            if not dfs_with_col:
                continue
            means, half_widths = compute_topk_curve(
                dfs_with_col, rank_col, SUMMARY_K_VALUES
            )
            row = {
                "comparison": comparison,
                "metric": metric,
                "model": label,
                "n_seeds": len(dfs_with_col),
            }
            for k, mean, hw in zip(SUMMARY_K_VALUES, means, half_widths):
                row[f"top-{k}"] = f"{mean:.1f} ± {hw:.1f}"
            correct_ranks = _correct_ranks(dfs_with_col[0], rank_col)
            row["MRR"] = f"{(1.0 / correct_ranks).mean():.4f}"
            row["median_rank"] = f"{correct_ranks.median():.1f}"
            rows.append(row)

df_summary = pd.DataFrame(rows)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_summary.to_csv(OUTPUT_DIR / "retrieval_formula_summary.csv", index=False)
df_summary

## Export summary to LaTeX

In [ ]:
def export_formula_summary_latex(
    df, output_path="figures/retrieval_formula/retrieval_formula_table.tex"
):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].str.replace("±", r"$\pm$", regex=False)
    latex = df.to_latex(index=False, escape=False)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        f.write(latex)
    print(f"LaTeX table exported to {output_path}")
    print(latex)


export_formula_summary_latex(df_summary)

In [ ]:
df_summary_rassp_subset = df_summary[
    df_summary["comparison"].isin(
        ["scaffold_rassp_subset", "random_rassp_subset"]
    )
]
df_summary_rassp_subset.to_csv(
    OUTPUT_DIR / "retrieval_formula_summary_rassp_subset.csv", index=False
)
export_formula_summary_latex(
    df_summary_rassp_subset,
    output_path="figures/retrieval_formula/retrieval_formula_table_rassp_subset.tex",
)
df_summary_rassp_subset

## RASSP-subset table (standalone)

Same rows as the "scaffold_rassp_subset" and "random_rassp_subset"
comparisons above, pulled into their own table + LaTeX export for direct
use in the RASSP head-to-head section of the paper.


## Inspect best, worst, and average retrieval cases

### ICICLE model
Loads pre-computed candidate spectra from `retrieval_with_formula_spectra.hdf5`.
Set `INSPECT_RESULTS_DIR` and `INSPECT_RANKING_METRIC` below.

### Baseline models (NEIMS / RASSP / MassFormer)
Requires two HDF5 files from `baselines/<model>/results/predictions/`:
- `*_test.hdf5` — predicted spectra for test queries (keyed by spec_id; attrs: `inchi_key`, `smiles`)
- `*_pubchem_cands_50.hdf5` — predicted spectra for PubChem candidates (keyed by int index; attrs: `inchi_key`, `smiles`)

GT spectrum is read from the NIST spectra HDF5.
Each panel: GT (top) vs top-1 predicted candidate. Title shows correct-answer rank, query SMILES, top-1 candidate SMILES, and whether top-1 is correct or a decoy.

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from icicle.utils.visualization.mass_spectra import plot_mirrored_spectra
from icicle.utils.visualization.style import (
    FIGSIZE,
    get_palette,
    save_fig,
    set_style,
    spec_colors,
)

set_style("manuscript")
palette = get_palette()

# ── Config ─────────────────────────────────────────────────────────────────
INSPECT_RESULTS_DIR = RESULTS / "final_entropy_scaffold_s2_retr"

INSPECT_RANKING_METRIC = "cosine_similarity"
N_EXAMPLES = 3

# ── Load data ───────────────────────────────────────────────────────────────
retrieval_csv = pd.read_csv(
    INSPECT_RESULTS_DIR / "retrieval_with_formula_results.csv"
)
hdf5_path = INSPECT_RESULTS_DIR / "retrieval_with_formula_spectra.hdf5"

rank_col = f"rank_{INSPECT_RANKING_METRIC}"
qcol = _query_col(retrieval_csv)

# Per-query: row of the correct candidate with the best (lowest) rank
try:
    correct_rows = retrieval_csv[~retrieval_csv["is_decoy"]].copy()
except KeyError:
    correct_rows = retrieval_csv[retrieval_csv["is_correct"]].copy()
best_correct_idx = correct_rows.groupby(qcol)[rank_col].idxmin()
query_correct_rank = correct_rows.loc[best_correct_idx].reset_index(drop=True)
query_correct_rank = query_correct_rank.sort_values(rank_col).reset_index(
    drop=True
)

n = len(query_correct_rank)
mid = n // 2
groups = {
    "Best (rank 1)": query_correct_rank.head(N_EXAMPLES),
    "Average (median rank)": query_correct_rank.iloc[
        mid - N_EXAMPLES // 2 : mid + N_EXAMPLES // 2 + 1
    ],
    "Worst (highest rank)": query_correct_rank.tail(N_EXAMPLES).iloc[::-1],
}

# ── Plot ─────────────────────────────────────────────────────────────────────
with h5py.File(hdf5_path, "r") as f:
    for group_name, subset in groups.items():
        print(f"\n{'=' * 60}")
        print(f"{group_name}")
        print(f"{'=' * 60}")
        for _, row in subset.iterrows():
            spec_id = str(int(row[qcol]))
            if spec_id not in f:
                print(f"  spec_id={spec_id} not in HDF5, skipping")
                continue

            grp = f[spec_id]
            gt_spec = grp["ground_truth_intensities"][:]
            cand_pred = grp["candidate_predicted_intensities"][:]
            cand_smiles = grp["candidate_smiles"].asstr()[:]
            cand_is_decoy = grp["candidate_is_decoy"][:]
            cand_ranks = grp[f"candidate_{rank_col}"][:]
            cand_cosine = grp["candidate_cosine_similarity"][:]

            top1_idx = int(np.argmin(cand_ranks))
            top1_pred = cand_pred[top1_idx]
            top1_smiles = cand_smiles[top1_idx]
            top1_is_decoy = bool(cand_is_decoy[top1_idx])
            sim_of_top1 = float(cand_cosine[top1_idx])

            correct_mask = ~cand_is_decoy.astype(bool)
            correct_idx = np.where(correct_mask)[0]
            correct_rank = (
                int(cand_ranks[correct_idx].min()) if len(correct_idx) else -1
            )
            gt_smiles = (
                cand_smiles[correct_idx[0]] if len(correct_idx) else "unknown"
            )
            sim_of_true = (
                float(cand_cosine[correct_idx[0]]) if len(correct_idx) else 0.0
            )

            if top1_pred.max() > 0:
                top1_pred = top1_pred / top1_pred.max()
            if gt_spec.max() > 0:
                gt_spec = gt_spec / gt_spec.max()

            top1_label = "correct" if not top1_is_decoy else "DECOY"
            n_decoys = int((cand_is_decoy == 1).sum())
            n_total = len(cand_smiles)

            title = (
                # f"{group_name} | correct rank={correct_rank} | {top1_label} @ top-1\n"
                # f"GT: {gt_smiles[:70]}\n"
                # f"Top-1: {top1_smiles[:70]}\n"
                # f"Candidates: {n_total} ({n_decoys} decoys)\n"
                # f"Similarity: top-1={sim_of_top1:.2f} | correct={sim_of_true:.2f}"
                ""
            )
            fig = plot_mirrored_spectra(
                true_spec=gt_spec,
                pred_spec=top1_pred,
                true_smiles=gt_smiles,
                title=title,
            )
            plt.show()
            plt.close(fig)

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from icicle.utils.visualization.mass_spectra import plot_mirrored_spectra
from icicle.utils.visualization.style import (
    FIGSIZE,
    get_palette,
    save_fig,
    set_style,
)

set_style("manuscript")

# ── Config ─────────────────────────────────────────────────────────────────
BASELINE_RETRIEVAL_CSV = Path(
    "/home/magled/icicle-dev/results/eval/rassp_scaffold_s1/retrieval_with_formula_results.csv"
)
BASELINE_TEST_HDF5 = Path(
    "/home/magled/icicle-dev/baselines/rassp/results/predictions/rassp_scaffold_s1_test.hdf5"
)
BASELINE_CANDS_HDF5 = Path(
    "/home/magled/icicle-dev/baselines/rassp/results/predictions/rassp_scaffold_s1_pubchem_cands_50.hdf5"
)
GT_SPECTRA_HDF5 = Path(
    "/home/magled/icicle-dev/data/NIST2023_GCMS_main/spectra.hdf5"
)
GT_METADATA_TSV = Path(
    "/home/magled/icicle-dev/data/NIST2023_GCMS_main/metadata.tsv"
)

BASELINE_RANKING_METRIC = "cosine_similarity"
N_EXAMPLES = 3
N_BINS = 1000  # RASSP bin count (m/z 1–1000)

# ── Build ik14 → candidate index map from pubchem cands file ─────────────────
print("Building ik14 → candidate index map (this may take a moment)...")
ik14_to_cand_idx = {}
with h5py.File(BASELINE_CANDS_HDF5, "r") as fc:
    for idx_str in fc.keys():
        full_ik = fc[idx_str].attrs["inchi_key"]
        ik14_to_cand_idx[full_ik[:14]] = idx_str
print(f"  {len(ik14_to_cand_idx):,} candidate entries indexed")

# ── Build lookup maps from NIST metadata ─────────────────────────────────────
# GT HDF5 keyed by mol_id (int string); each entry: group with masses + intensities
meta = pd.read_csv(
    GT_METADATA_TSV,
    sep="\t",
    usecols=["mol_id", "inchikey", "standardized_smiles"],
)
meta["inchikey14"] = meta["inchikey"].str[:14]
ik14_to_molid = dict(zip(meta["inchikey14"], meta["mol_id"].astype(str)))
ik14_to_smiles = dict(zip(meta["inchikey14"], meta["standardized_smiles"]))

# ── Load retrieval CSV and find best/worst/average by correct-answer rank ───
retrieval_csv = pd.read_csv(BASELINE_RETRIEVAL_CSV)
rank_col = f"rank_{BASELINE_RANKING_METRIC}"
qcol = _query_col(retrieval_csv)

correct_rows = retrieval_csv[retrieval_csv["is_correct"]].copy()
best_correct_idx = correct_rows.groupby(qcol)[rank_col].idxmin()
query_correct_rank = correct_rows.loc[best_correct_idx].reset_index(drop=True)
query_correct_rank = query_correct_rank.sort_values(rank_col).reset_index(
    drop=True
)

n = len(query_correct_rank)
mid = n // 2
groups = {
    "Best (rank 1)": query_correct_rank.head(N_EXAMPLES),
    "Average (median rank)": query_correct_rank.iloc[
        mid - N_EXAMPLES // 2 : mid + N_EXAMPLES // 2 + 1
    ],
    "Worst (highest rank)": query_correct_rank.tail(N_EXAMPLES).iloc[::-1],
}


def _sparse_to_dense(masses, intensities, n_bins):
    """Convert sparse (masses, intensities) to dense binned array."""
    dense = np.zeros(n_bins, dtype=np.float32)
    idx = np.round(masses).astype(int) - 1  # m/z 1 → bin 0
    valid = (idx >= 0) & (idx < n_bins)
    dense[idx[valid]] = intensities[valid]
    return dense

In [ ]:
# ── Plot ─────────────────────────────────────────────────────────────────────
with (
    h5py.File(BASELINE_TEST_HDF5, "r") as ft,
    h5py.File(BASELINE_CANDS_HDF5, "r") as fc,
    h5py.File(GT_SPECTRA_HDF5, "r") as fg,
):
    for group_name, subset in groups.items():
        print(f"\n{'=' * 60}")
        print(f"{group_name}")
        print(f"{'=' * 60}")
        for _, row in subset.iterrows():
            query_ik14 = str(row[qcol])
            correct_rank = int(row[rank_col])

            mol_id = ik14_to_molid.get(query_ik14)
            if mol_id is None or mol_id not in fg:
                print(
                    f"  query={query_ik14} (mol_id={mol_id}): GT not found, skipping"
                )
                continue

            gt_grp = fg[mol_id]
            gt_spec = _sparse_to_dense(
                gt_grp["masses"][:], gt_grp["intensities"][:], N_BINS
            )
            query_smiles = ik14_to_smiles.get(query_ik14, "unknown")

            # top-1 candidate by ranking metric
            query_rows = retrieval_csv[retrieval_csv[qcol] == query_ik14]
            top1_row = query_rows.loc[query_rows[rank_col].idxmin()]
            top1_ik14 = top1_row["candidate_inchikey14"]
            top1_is_correct = bool(top1_row["is_correct"])
            n_candidates = len(query_rows)
            n_decoys = int((~query_rows["is_correct"]).sum())

            cand_idx = ik14_to_cand_idx.get(top1_ik14)
            if cand_idx is None:
                print(f"  top-1 ik14 {top1_ik14} not in cands HDF5, skipping")
                continue

            top1_pred = fc[cand_idx]["predicted_intensities"][:]
            top1_smiles = fc[cand_idx].attrs.get("smiles", "unknown")

            if top1_pred.max() > 0:
                top1_pred = top1_pred / top1_pred.max()
            if gt_spec.max() > 0:
                gt_spec = gt_spec / gt_spec.max()

            top1_label = "correct" if top1_is_correct else "DECOY"
            title = (
                f"{group_name} | correct rank={correct_rank} | {top1_label} @ top-1\n"
                f"GT: {query_smiles[:70]}\n"
                f"Top-1: {top1_smiles[:70]}\n"
                f"Candidates: {n_candidates} ({n_decoys} decoys)"
            )
            fig = plot_mirrored_spectra(
                true_spec=gt_spec,
                pred_spec=top1_pred,
                true_smiles=query_smiles,
                title=title,
            )
            plt.show()
            plt.close(fig)